<a href="https://colab.research.google.com/github/BhumiPatil1507/BTP-/blob/main/Simile_to_Image_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install rdkit

In [ ]:
import pandas as pd
import os
import time
from rdkit import Chem
from rdkit.Chem import Draw

# ============================================================
# 1. SETTINGS
# ============================================================

CSV_PATH = '/content/drive/MyDrive/Colab/polyOne_aa.csv'

# Save images directly to Google Drive
OUTPUT_DIR = '/content/drive/MyDrive/Colab/polymer_images'

# Number of molecules processed in one batch
BATCH_SIZE = 5000

# Image size
IMAGE_SIZE = (300, 300)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

TOTAL = len(df)

print("=" * 60)
print("POLYMER IMAGE GENERATION")
print("=" * 60)
print(f"Total molecules: {TOTAL:,}")
print(f"Batch size:       {BATCH_SIZE:,}")
print(f"Output folder:    {OUTPUT_DIR}")
print("=" * 60)


# ============================================================
# 3. FIND ALREADY COMPLETED IMAGES
# ============================================================

existing_images = set(os.listdir(OUTPUT_DIR))

completed_indices = set()

for filename in existing_images:
    if filename.startswith("polymer_") and filename.endswith(".png"):
        try:
            number = int(filename[8:-4])
            completed_indices.add(number)
        except:
            pass

print(f"Already completed: {len(completed_indices):,}")
print(f"Remaining:         {TOTAL - len(completed_indices):,}")


# ============================================================
# 4. PROCESS IN BATCHES
# ============================================================

overall_start = time.time()

total_generated = 0
total_skipped = 0
total_invalid = 0

for batch_start in range(0, TOTAL, BATCH_SIZE):

    batch_end = min(batch_start + BATCH_SIZE, TOTAL)

    print("\n" + "=" * 60)
    print(f"BATCH: {batch_start:,} → {batch_end - 1:,}")
    print("=" * 60)

    batch_records = []

    batch_start_time = time.time()

    for i in range(batch_start, batch_end):

        # ----------------------------------------------------
        # Skip image if it already exists
        # ----------------------------------------------------

        image_name = f"polymer_{i:06d}.png"
        image_path = os.path.join(OUTPUT_DIR, image_name)

        if os.path.exists(image_path):
            total_skipped += 1

            continue


        # ----------------------------------------------------
        # Get SMILES
        # ----------------------------------------------------

        smiles = df.iloc[i]["smiles"]


        # ----------------------------------------------------
        # Convert SMILES → RDKit molecule
        # ----------------------------------------------------

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            total_invalid += 1

            continue


        # ----------------------------------------------------
        # Generate image
        # ----------------------------------------------------

        try:

            img = Draw.MolToImage(
                mol,
                size=IMAGE_SIZE
            )

            img.save(image_path)

        except Exception as e:

            print(f"\nError at molecule {i}: {e}")
            continue


        # ----------------------------------------------------
        # Save mapping information
        # ----------------------------------------------------

        batch_records.append({
            "image_id": image_name,
            "smiles": smiles
        })

        total_generated += 1


        # ----------------------------------------------------
        # Progress display
        # ----------------------------------------------------

        processed = i + 1

        percentage = (processed / TOTAL) * 100

        elapsed = time.time() - overall_start

        if total_generated > 0:

            time_per_molecule = elapsed / total_generated

            remaining_molecules = TOTAL - len(completed_indices) - total_generated

            estimated_seconds = (
                remaining_molecules * time_per_molecule
            )

            hours = int(estimated_seconds // 3600)
            minutes = int((estimated_seconds % 3600) // 60)
            seconds = int(estimated_seconds % 60)

            eta = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

        else:

            eta = "calculating..."


        # Print every 100 newly generated images

        if total_generated % 100 == 0:

            print(
                f"\rProgress: {percentage:6.2f}% | "
                f"Processed: {processed:,}/{TOTAL:,} | "
                f"New images: {total_generated:,} | "
                f"Invalid: {total_invalid:,} | "
                f"ETA: {eta}",
                end=""
            )


    # ========================================================
    # 5. SAVE MAPPING FOR THIS BATCH
    # ========================================================

    if batch_records:

        batch_mapping = pd.DataFrame(batch_records)

        mapping_path = os.path.join(
            OUTPUT_DIR,
            f"mapping_{batch_start:06d}_{batch_end - 1:06d}.csv"
        )

        batch_mapping.to_csv(
            mapping_path,
            index=False
        )

        print(
            f"\n\nMapping saved: "
            f"{os.path.basename(mapping_path)}"
        )


    # ========================================================
    # 6. BATCH SUMMARY
    # ========================================================

    batch_time = time.time() - batch_start_time

    print(
        f"Batch completed in "
        f"{batch_time / 60:.2f} minutes"
    )

    print(
        f"New images generated: {len(batch_records):,}"
    )

    print(
        f"Invalid SMILES: {total_invalid:,}"
    )


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

total_time = time.time() - overall_start

print("\n")
print("=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)

print(f"Total dataset:       {TOTAL:,}")
print(f"New images generated:{total_generated:,}")
print(f"Already existed:     {total_skipped:,}")
print(f"Invalid SMILES:      {total_invalid:,}")

print(
    f"Total time:          "
    f"{total_time / 3600:.2f} hours"
)

print("=" * 60)

In [ ]:
import pandas as pd
import os
import time
from rdkit import Chem
from rdkit.Chem import Draw

# ============================================================
# 1. SETTINGS
# ============================================================

CSV_PATH = '/content/drive/MyDrive/Colab/polyOne_aa.csv'

OUTPUT_DIR = '/content/drive/MyDrive/Colab/polymer_images'

BATCH_SIZE = 5000

IMAGE_SIZE = (300, 300)

# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

TOTAL = len(df)

print("=" * 60)
print("RESUMING POLYMER IMAGE GENERATION")
print("=" * 60)
print(f"Total molecules: {TOTAL:,}")
print(f"Output folder:   {OUTPUT_DIR}")
print("=" * 60)

# ============================================================
# 3. CHECK DRIVE
# ============================================================

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Output directory accessible.")

# ============================================================
# 4. FIND MISSING IMAGES
# ============================================================

print("\nChecking for missing images...")
print("This may take some time because Drive contains many files.")

missing_indices = []

for i in range(TOTAL):

    image_name = f"polymer_{i:06d}.png"
    image_path = os.path.join(OUTPUT_DIR, image_name)

    if not os.path.exists(image_path):
        missing_indices.append(i)

print("\n" + "=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

print(f"Total molecules:   {TOTAL:,}")
print(f"Existing images:   {TOTAL - len(missing_indices):,}")
print(f"Missing images:    {len(missing_indices):,}")
print("=" * 60)

# ============================================================
# 5. PROCESS ONLY MISSING IMAGES
# ============================================================

if len(missing_indices) == 0:

    print("\nAll images already exist!")
    print("Nothing to generate.")

else:

    overall_start = time.time()

    generated = 0
    invalid = 0
    errors = 0

    total_missing = len(missing_indices)

    print(f"\nStarting generation of {total_missing:,} missing images...\n")

    for count, i in enumerate(missing_indices, start=1):

        # ----------------------------------------------------
        # Image path
        # ----------------------------------------------------

        image_name = f"polymer_{i:06d}.png"
        image_path = os.path.join(OUTPUT_DIR, image_name)

        # ----------------------------------------------------
        # Get SMILES
        # ----------------------------------------------------

        smiles = df.iloc[i]["smiles"]

        # ----------------------------------------------------
        # Convert SMILES → molecule
        # ----------------------------------------------------

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:

            invalid += 1

            print(
                f"\nInvalid SMILES at index {i}: {smiles}"
            )

            continue

        # ----------------------------------------------------
        # Generate image
        # ----------------------------------------------------

        try:

            img = Draw.MolToImage(
                mol,
                size=IMAGE_SIZE
            )

            img.save(image_path)

            generated += 1

        except Exception as e:

            errors += 1

            print(
                f"\nError at index {i}: {e}"
            )

            continue

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if count % 100 == 0 or count == total_missing:

            elapsed = time.time() - overall_start

            rate = generated / elapsed if elapsed > 0 else 0

            remaining = total_missing - count

            eta_seconds = (
                remaining / rate
                if rate > 0
                else 0
            )

            hours = int(eta_seconds // 3600)

            minutes = int(
                (eta_seconds % 3600) // 60
            )

            seconds = int(
                eta_seconds % 60
            )

            print(
                f"\rProgress: {count:,}/{total_missing:,} "
                f"({count / total_missing * 100:6.2f}%) | "
                f"Generated: {generated:,} | "
                f"Invalid: {invalid:,} | "
                f"Errors: {errors:,} | "
                f"ETA: {hours:02d}:{minutes:02d}:{seconds:02d}",
                end=""
            )

    # ========================================================
    # 6. FINAL SUMMARY
    # ========================================================

    total_time = time.time() - overall_start

    print("\n\n" + "=" * 60)
    print("RESUME GENERATION COMPLETE")
    print("=" * 60)

    print(f"Previously existing: {TOTAL - total_missing:,}")
    print(f"Missing at start:    {total_missing:,}")
    print(f"Newly generated:     {generated:,}")
    print(f"Invalid SMILES:      {invalid:,}")
    print(f"Errors:              {errors:,}")

    print(
        f"Total time:          "
        f"{total_time / 3600:.2f} hours"
    )

    print("=" * 60)

In [ ]:
folder = "/content/drive/MyDrive/Colab/polymer_images"

In [ ]:
import os

print(os.path.exists(folder))

In [ ]:
mport os

folder = "/content/drive/MyDrive/Colab/polymer_images"

count = 0

for root, dirs, files in os.walk(folder):
    count += sum(
        f.lower().endswith((".png", ".jpg", ".jpeg"))
        for f in files
    )

print("Total polymer images:", count)
print("Missing:", 500000 - count)